# RugbyVision — Demo Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/rugby-vision/blob/main/notebooks/demo.ipynb)

This notebook walks through the full RugbyVision pipeline:
1. Install dependencies
2. Download a sample clip
3. Run detection + tracking + team classification
4. Project to 2-D field view
5. Render heatmap and distance chart


In [ ]:
# @title Install RugbyVision dependencies
!pip install -q ultralytics supervision opencv-python-headless scikit-learn matplotlib yt-dlp pyyaml
# Clone repo (skip if already mounted)
import os
if not os.path.exists('rugby-vision'):
    !git clone https://github.com/YOUR_USERNAME/rugby-vision.git
%cd rugby-vision

In [ ]:
import sys, yaml, cv2, numpy as np
from IPython.display import Image, display
from pathlib import Path

sys.path.insert(0, '.')
from rugby_vision.detector import PlayerDetector
from rugby_vision.tracker import PlayerTracker
from rugby_vision.team_classifier import TeamClassifier
from rugby_vision.homography import FieldHomography
from rugby_vision.visualizer import Visualizer
from rugby_vision.metrics import MetricsCollector

cfg = yaml.safe_load(open('config.yaml'))
print('Config loaded ✓')

In [ ]:
# @title Download a sample rugby clip from YouTube
VIDEO_URL = 'https://www.youtube.com/watch?v=REPLACE_ME'  # @param {type:"string"}
!yt-dlp -f mp4 -o sample.mp4 "{VIDEO_URL}"

In [ ]:
VIDEO_PATH = 'sample.mp4'

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
N   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {W}×{H} @ {fps:.1f} fps  ({N} frames)')

detector = PlayerDetector.from_config(cfg['model'])
tracker  = PlayerTracker.from_config(cfg['tracking'], frame_rate=int(fps))
clf      = TeamClassifier.from_config(cfg['team_classifier'])
metrics  = MetricsCollector.from_config(cfg, fps=fps)
vis      = Visualizer.from_config(cfg)
print('Modules initialised ✓')

In [ ]:
# @title Manual homography calibration
# In Colab we can't use an interactive OpenCV window.
# Instead, display the first frame and enter corner coordinates manually.

ret, first_frame = cap.read()
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# Save first frame for inspection
cv2.imwrite('first_frame.jpg', first_frame)
display(Image('first_frame.jpg'))
print('Look at the image above and note the pixel (x, y) of the 4 field corners.')
print('Order: Top-Left, Top-Right, Bottom-Right, Bottom-Left')

In [ ]:
# @title Enter corner pixel coordinates
# Replace with your actual values after inspecting the frame above
corners = np.array([
    [120, 80],   # Top-Left
    [900, 80],   # Top-Right
    [900, 620],  # Bottom-Right
    [120, 620],  # Bottom-Left
], dtype=np.float32)

hom = FieldHomography(
    field_length_m=cfg['homography']['field_length_m'],
    field_width_m=cfg['homography']['field_width_m'],
)
hom.calibrate_from_points(corners)
print('Homography calibrated ✓')

In [ ]:
# @title Warm-up: fit team classifier on first 30 frames
WARMUP = 30
frames_w, dets_w = [], []
for _ in range(WARMUP):
    ret, frame = cap.read()
    if not ret: break
    d = detector.detect(frame)
    if len(d) > 0:
        frames_w.append(frame)
        dets_w.append(d)

if frames_w:
    clf.fit(frames_w, dets_w)
    print(f'TeamClassifier fitted on {len(frames_w)} frames ✓')
else:
    print('No detections during warm-up')
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

In [ ]:
# @title Run the full pipeline
MAX_FRAMES = 300  # @param {type:"integer"}

Path('outputs').mkdir(exist_ok=True)
field_dims = (cfg['homography']['field_width_m'], cfg['homography']['field_length_m'])

mm_cfg = cfg['visualization']['minimap']
out_w = W + int(mm_cfg['width'] * (H / mm_cfg['height']))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter('outputs/annotated.mp4', fourcc, fps, (out_w, H))

for i in range(MAX_FRAMES):
    ret, frame = cap.read()
    if not ret: break

    dets        = detector.detect(frame)
    dets        = tracker.update(dets, frame)
    team_labels = clf.predict(frame, dets) if clf.is_fitted() and len(dets) > 0 else None
    field_pts   = hom.project_detections(dets) if len(dets) > 0 else None

    if field_pts is not None and dets.tracker_id is not None:
        metrics.update(dets.tracker_id, field_pts, team_labels)

    out_frame = vis.draw(frame, dets, team_labels, field_pts, field_dims)
    writer.write(out_frame)

    if i % 50 == 0:
        print(f'  frame {i}/{MAX_FRAMES}')

cap.release()
writer.release()
print('Pipeline done ✓  →  outputs/annotated.mp4')

In [ ]:
# @title Metrics and visualisations
metrics.plot_heatmap('outputs/heatmap.png', title='All Players Heatmap')
metrics.plot_distances('outputs/distances.png')

display(Image('outputs/heatmap.png'))
display(Image('outputs/distances.png'))

summary = metrics.summary()
print(f'\n{len(summary)} players tracked')
for tid, s in sorted(summary.items()):
    print(f'  ID {tid:>3} | team {s["team"]} | {s["distance_m"]:6.1f} m | {s["speed_ms"]:.1f} m/s')